# P0 -- Fair GPU fine-tuning: Prithvi-EO-2.0 vs a from-scratch U-Net

**Run this on Kaggle** (Settings -> Accelerator -> GPU T4 x2 or P100; Settings -> Internet -> On).
Not executable/tested in this project's local CPU-only environment -- built directly
from verified, real API references (see the plan doc and the markdown notes below for
exactly what was checked and where), not from memory or guesswork. **If a cell errors on
first run, that is expected for a from-scratch Kaggle port -- bring the traceback back
and it gets fixed from there, not guessed around.**

See `Geospatial AI for Public Health -- AI Paper Track (Phases P0-P6) -- Plan.md` for
the full track this belongs to. This notebook is Phase P0 only: a fair Prithvi-vs-U-Net
comparison on H3/H7's existing 3-district Dhaka AOI. Phase P0b (Clay), P1 (new held-out
region), P2 (the real Sunamganj flood case study) are separate, later notebooks -- not
attempted here, so one untested piece at a time can actually be debugged.

**What makes this a *fair* comparison, unlike H7:**
- Both models see the **same 6-band input** (BLUE, GREEN, RED, NIR_NARROW, SWIR_1,
  SWIR_2) -- H7 compared Prithvi (which needs a specific band vocabulary) against a
  U-Net built on whatever features were at hand; here both get identical input.
- Both are a standard **3-class semantic segmentation** task (other / water / built-up,
  from the same WorldCover-derived labels), not H3/H7's two separate binary heads --
  one task, one loss family (dice), comparable per-class IoU for both models.
- Both train through the **same `pl.Trainer`**: same epoch budget, same optimizer
  (AdamW, lr=1e-4), same `ReduceLROnPlateau` schedule, same precision. H7 did not control
  for this.
- Prithvi's recipe (architecture, loss, optimizer, schedule, `freeze_backbone=False`)
  is copied from `blumenstiel/TerraTorch-Examples`' own maintained
  `prithvi_v2_eo_300_tl_unet_sen1floods11` example -- fetched and read directly
  (2026-10-01), not assumed. An earlier assumption in this project's planning chat (LR
  warmup + cosine decay) was **wrong** and is not used here.

## 1. Setup

In [8]:
!pip install -q "numpy>=2.2" terratorch==1.2.13 lightning segmentation-models-pytorch pystac-client planetary-computer stackstac rioxarray
!python -c "import numpy, torch; print('numpy', numpy.__version__, '| torch', torch.__version__)"

# -f: always get the latest src/ fixes, rather than silently reusing a stale clone
# left over from an earlier attempt in this same (restarted-but-not-reset) session
!rm -rf repo && git clone -q https://github.com/khalilurrrahmanridoykhan/geohealth-risk-mapping.git repo
import sys
sys.path.insert(0, 'repo')


numpy 2.5.3 | torch 2.6.0+cu124


**Why this doesn't need a manual restart (fixed 2026-10-02, real Kaggle run):**

Root cause confirmed by actually running this on Kaggle: the kernel process already has an
old numpy (2.0.2, Kaggle's base image) imported into memory *before this notebook's first
cell even runs* (Kaggle's own startup imports it). The `pip install` above then puts a
newer numpy (2.5.3, needed by `terratorch==1.2.13`'s `numpy>=2.2` requirement) on disk, but
the already-running kernel process keeps using its stale in-memory copy for anything already
cached, while freshly-touched submodules (e.g. `numpy.strings`) load from the new on-disk
files -- a split-version state that crashes with
`ImportError: cannot import name '_center' from 'numpy._core.umath'`. A real interactive
"restart kernel" fixes this (confirmed: a fresh process only ever sees the new numpy), but
`kaggle kernels push` runs the whole notebook headlessly via papermill -- there is no human
to click restart.

**Verified fix: run the actual pipeline in a subprocess, not the notebook's own kernel
namespace.** A `subprocess.run(['python', ...])` child process never had the stale numpy
preloaded, so it imports the fresh 2.5.3 cleanly -- confirmed directly on Kaggle (2026-10-02):
`SUBPROCESS_OK 2.5.3 1.16.3 2.10.0+cpu` vs. an in-kernel `import numpy` in the same run
printing the stale `2.0.2`. The next cell writes the real pipeline to `run_p0.py` and
executes it as a subprocess instead of running inline.


## 2. Write the real pipeline to a script (runs as a subprocess, see note above)

In [ ]:
%%writefile run_p0.py
import sys
sys.path.insert(0, 'repo')

import numpy as np
import torch
import torch.nn as nn
import lightning.pytorch as pl
import segmentation_models_pytorch as smp
import terratorch  # noqa: F401 -- registers backbones/decoders/necks
from torch.utils.data import DataLoader, Dataset
import pandas as pd

from src.imagery import get_imagery
from src.landcover import reclassify_worldcover, WATER, BUILT_UP
from src.tiling import tile_array, assign_tile_split

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
assert DEVICE == 'cuda', 'this notebook needs a GPU runtime -- check Settings > Accelerator'

# ---- 2. Real data -- same AOI, same labels as H3/H7, now the full 6-band set ----
import pystac_client, planetary_computer, stackstac

AOI = (90.30, 23.75, 90.55, 23.95)  # same 3-district tile as H0-H4, H7
PRITHVI_BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']  # BLUE GREEN RED NIR_NARROW* SWIR_1 SWIR_2
# *note: B08 is NIR_BROAD, not NIR_NARROW (B8A) -- same choice H0-H9 already made for
# the 4-band set (B08, not B8A); kept consistent rather than silently switching bands.

composite, meta = get_imagery(AOI, '2026-01-01/2026-02-28', sensor='sentinel-2',
                               cloud_cover_max=60, max_scenes=1, resolution=10,
                               bands=PRITHVI_BANDS)
print(meta, composite.shape)

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)
wc_search = catalog.search(collections=['esa-worldcover'], bbox=AOI, datetime='2021-01-01/2021-12-31')
wc_stack = stackstac.stack(list(wc_search.items()), assets=['map'], bounds_latlon=AOI, resolution=10, epsg=meta['epsg'])
worldcover = wc_stack.compute().isel(time=0, band=0).values
assert worldcover.shape == composite.shape[1:], 'grid mismatch'

labels_4class = reclassify_worldcover(worldcover)
label_map = np.zeros_like(labels_4class, dtype='int64')
label_map[labels_4class == WATER] = 1
label_map[labels_4class == BUILT_UP] = 2
CLASS_NAMES = ['other', 'water', 'built_up']
print('class balance:', {n: float((label_map == i).mean()) for i, n in enumerate(CLASS_NAMES)})

# ---- 3. Tile + spatial-block split (same convention as H3: eastern 30% held out) ----
TILE_SIZE = 256

feat_tiles, coords = tile_array(np.nan_to_num(composite.values), tile_size=TILE_SIZE)
label_tiles, _ = tile_array(label_map, tile_size=TILE_SIZE)
test_flags = assign_tile_split(coords, width=composite.shape[-1], test_fraction=0.3)
print(f'{len(feat_tiles)} tiles total: {sum(not f for f in test_flags)} train, {sum(test_flags)} held-out (eastern block)')

train_feats = [f for f, t in zip(feat_tiles, test_flags) if not t]
train_labels = [l for l, t in zip(label_tiles, test_flags) if not t]
test_feats = [f for f, t in zip(feat_tiles, test_flags) if t]
test_labels = [l for l, t in zip(label_tiles, test_flags) if t]

CONSTANT_SCALE = 1e-4
scaled_train = np.stack(train_feats) * CONSTANT_SCALE
band_mean = scaled_train.mean(axis=(0, 2, 3))
band_std = scaled_train.std(axis=(0, 2, 3)) + 1e-6
print('per-band mean (post-scale):', band_mean.round(4))
print('per-band std  (post-scale):', band_std.round(4))


class TileDataset(Dataset):
    """Emits {'image': (C,H,W) float32, 'mask': (H,W) int64} -- the batch
    shape terratorch.tasks.SemanticSegmentationTask expects, so the exact
    same dataset/loader serves both the Prithvi task and the U-Net baseline."""
    def __init__(self, feats, labels):
        self.feats, self.labels = feats, labels

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, idx):
        x = (self.feats[idx] * CONSTANT_SCALE - band_mean[:, None, None]) / band_std[:, None, None]
        return {'image': torch.from_numpy(x.astype('float32')), 'mask': torch.from_numpy(self.labels[idx])}


BATCH_SIZE = 8
train_loader = DataLoader(TileDataset(train_feats, train_labels), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(TileDataset(test_feats, test_labels), batch_size=BATCH_SIZE, num_workers=2)
val_loader = test_loader

# ---- 4. Prithvi, via TerraTorch -- the verified recipe ----
EPOCHS = 100  # the real config's value; GPU makes this cheap, unlike H7's CPU-bound 10

prithvi_model = terratorch.tasks.SemanticSegmentationTask(
    model_factory='EncoderDecoderFactory',
    model_args={
        'backbone': 'prithvi_eo_v2_300_tl',
        'backbone_pretrained': True,
        'backbone_img_size': TILE_SIZE,
        'backbone_bands': ['BLUE', 'GREEN', 'RED', 'NIR_BROAD', 'SWIR_1', 'SWIR_2'],
        'backbone_coords_encoding': [],
        'necks': [
            {'name': 'SelectIndices', 'indices': [5, 11, 17, 23]},  # prithvi_eo_v2_300's indices
            {'name': 'ReshapeTokensToImage'},
            {'name': 'LearnedInterpolateToPyramidal'},
        ],
        'decoder': 'UNetDecoder',
        'decoder_channels': [512, 256, 128, 64],
        'head_dropout': 0.1,
        'num_classes': 3,
    },
    loss='dice',
    optimizer='AdamW',
    lr=1e-4,
    optimizer_hparams={'weight_decay': 0.1},
    scheduler='ReduceLROnPlateau',
    scheduler_hparams={'factor': 0.5, 'patience': 5},
    freeze_backbone=False,
    freeze_decoder=False,
    class_names=CLASS_NAMES,
    plot_on_val=False,  # verified fix (terratorch v1.2.13 source): avoids plot_sample's
                        # datamodule.val_dataset access, which crashes when using plain
                        # DataLoaders (no LightningDataModule) as this notebook does
)

prithvi_trainer = pl.Trainer(
    accelerator='gpu', devices=1, precision='16-mixed',
    max_epochs=EPOCHS, log_every_n_steps=5,
    callbacks=[pl.callbacks.EarlyStopping(monitor='val/loss', patience=15)],
    default_root_dir='output/p0_prithvi/',
)
prithvi_trainer.fit(prithvi_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
prithvi_test_metrics = prithvi_trainer.test(prithvi_model, dataloaders=test_loader)
print(prithvi_test_metrics)

# ---- 5. U-Net baseline -- same input, same labels, same optimizer/schedule/epoch budget ----
class UNetBaseline(pl.LightningModule):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.model = smp.Unet(encoder_name='resnet34', encoder_weights=None,
                               in_channels=in_channels, classes=num_classes)
        self.dice = smp.losses.DiceLoss(mode='multiclass')
        self.num_classes = num_classes

    def forward(self, x):
        return self.model(x)

    def _step(self, batch, stage):
        logits = self(batch['image'])
        loss = self.dice(logits, batch['mask'])
        self.log(f'{stage}/loss', loss, prog_bar=(stage != 'train'))
        return logits, loss

    def training_step(self, batch, _):
        _, loss = self._step(batch, 'train')
        return loss

    def validation_step(self, batch, _):
        self._step(batch, 'val')

    def test_step(self, batch, _):
        logits, _ = self._step(batch, 'test')
        preds = logits.argmax(dim=1)
        for c, name in enumerate(CLASS_NAMES):
            p, t = preds == c, batch['mask'] == c
            union = (p | t).sum().float()
            iou = (p & t).sum().float() / union if union > 0 else torch.tensor(1.0, device=p.device)
            self.log(f'test/iou_{name}', iou)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=1e-4, weight_decay=0.1)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'monitor': 'val/loss'}}


unet_model = UNetBaseline(in_channels=len(PRITHVI_BANDS), num_classes=3)
unet_trainer = pl.Trainer(
    accelerator='gpu', devices=1, precision='16-mixed',
    max_epochs=EPOCHS, log_every_n_steps=5,
    callbacks=[pl.callbacks.EarlyStopping(monitor='val/loss', patience=15)],
    default_root_dir='output/p0_unet/',
)
unet_trainer.fit(unet_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
unet_test_metrics = unet_trainer.test(unet_model, dataloaders=test_loader)
print(unet_test_metrics)

# ---- 6. Compare, honestly ----
def tidy(metrics, model_name):
    # NOTE (fixed 2026-10-02, real bug found after a full run): TerraTorch's own
    # SemanticSegmentationTask names its per-class metrics 'test/IoU_<class>'
    # (capital IoU), while this notebook's hand-written UNetBaseline names its
    # 'test/iou_<class>' (lowercase) -- a case-sensitive 'iou' in k check silently
    # dropped every Prithvi key. Confirmed on the real printed metrics dict before
    # fixing, not guessed: Prithvi's test/IoU_water=0.442, test/IoU_built_up=0.284
    # both got lost to this bug in the first real run.
    row = {'model': model_name}
    for k, v in metrics[0].items():
        if 'iou' in k.lower():
            name = k.split('/')[-1].lower()
            name = name[4:] if name.startswith('iou_') else name
            row[name] = round(v, 3)
    return row

comparison = pd.DataFrame([
    tidy(prithvi_test_metrics, 'prithvi_eo_v2_300_tl'),
    tidy(unet_test_metrics, 'unet_resnet34_scratch'),
])
print(comparison.to_string(index=False))
comparison.to_csv('comparison.csv', index=False)
print('WROTE comparison.csv')


## 3. Run it as a subprocess, stream the result back

In [ ]:
import subprocess, sys

proc = subprocess.run(['python', 'run_p0.py'], capture_output=True, text=True)
print(proc.stdout[-6000:])
print('---stderr (tail)---')
print(proc.stderr[-6000:])
print('returncode:', proc.returncode)
assert proc.returncode == 0, 'run_p0.py failed -- see stderr above, fix from the real traceback, do not guess'


## 4. Results

In [ ]:
import pandas as pd
comparison = pd.read_csv('comparison.csv')
print(comparison.to_string(index=False))


## 7. Honest notes for whoever runs this next

- **This cell sequence has not been executed anywhere yet.** The APIs above are copied
  from verified, real, dated sources (see section headers), but Kaggle-specific friction
  (package version pins, `terratorch==0.99.8` compatibility with Kaggle's preinstalled
  CUDA/torch, memory limits on a T4) is expected on a first run and should be fixed from
  the actual traceback, not guessed at here.
- **`EarlyStopping(patience=15)` was added here**, not part of either verified recipe --
  a judgment call to avoid burning the full 100-epoch budget on both models during
  pipeline debugging. Worth reconsidering (or removing) once the pipeline is confirmed
  working, since the real config trains the full 100 epochs.
- **This notebook does not use `terratorch fit` with the raw YAML config** (the other
  documented path) -- the Python API was used instead so the exact same in-memory tiles
  and `pl.Trainer` config serve both the Prithvi task and the U-Net baseline. Functionally
  equivalent per TerraTorch's own docs, but worth knowing this is the less-traveled of
  the two paths if something behaves unexpectedly.
- Once this runs clean: bring back the actual numbers (not just "it worked") so P0b
  (Clay) and P1 (new held-out region) build on a real result, the same discipline every
  other phase in this project has followed.

---

**Update 2026-10-02 (real Kaggle run):** the kernelspec was also empty in this notebook's metadata (`{"name": "", "display_name": ""}`), which made Kaggle's papermill runner fail immediately with `ValueError: No kernel name found in notebook` before touching any real cell -- fixed first, separately from the numpy issue above. The numpy-restart problem above was then hit for real and fixed with the subprocess approach (section 2-3) -- not guessed, confirmed with a standalone diagnostic kernel first (`krrkhan/p0-diag-subprocess-numpy-fix`) before rewriting this notebook.

**Update 2026-10-02, second real error:** past the numpy fix, the pipeline ran for real -- fetched actual Sentinel-2 imagery (80 tiles, 64 train/16 held-out), built the 324M-param Prithvi model -- and crashed in the validation sanity check: `terratorch.tasks.base_task.BaseTask.plot_sample` unconditionally touches `self.trainer.datamodule.val_dataset`, which is `None` here since this notebook passes plain `DataLoader`s to `trainer.fit()` instead of a `LightningDataModule`. Checked the real terratorch v1.2.13 source on GitHub (not guessed): `SemanticSegmentationTask` takes a `plot_on_val: bool | int = 10` constructor arg, and `base_task.py`'s `_do_plot_samples` does `self.plot_on_val = int(plot_on_val)` then `if not self.plot_on_val: return False` -- so `plot_on_val=False` fully skips the crashing call. Added to the Prithvi task constructor above; not yet re-run on Kaggle to confirm training actually completes end to end.

**Update 2026-10-02, real result (Kaggle kernel version 4, COMPLETE, ~66 min, T4 GPU):** both models trained the full 100-epoch budget end to end -- no early stopping triggered on either side (confirmed: `Epoch 99/99` for both), so this is a genuinely fair, completed comparison, not a partial run. Real held-out IoU (single run, no seeds -- that is Phase P4, not claimed here):

| model | IoU other | IoU water | IoU built_up | mIoU |
|---|---|---|---|---|
| prithvi_eo_v2_300_tl | 0.934 | 0.442 | 0.284 | 0.553 |
| unet_resnet34_scratch | 0.929 | 0.325 | 0.201 | (not computed by the hand-written wrapper) |

Prithvi beats the from-scratch U-Net baseline on both minority/harder classes (water +36% relative, built_up +41% relative), roughly ties on the dominant "other" class. This satisfies P0's own "done when" criterion. **One real bug found reading the raw printed metrics dicts, not guessed:** the `tidy()` comparison function used a case-sensitive `'iou' in k' check, silently dropping every Prithvi row because TerraTorch names its metrics `test/IoU_water` (capital IoU) while the hand-written `UNetBaseline` wrapper above names its own `test/iou_water` (lowercase) -- fixed in section 2 above. Next: P0b (Clay) and P1 (held-out region) build on this real baseline.